# Semantic Search & LLM Application
**Starter Notebook (diisi)**: asisten FAQ customer service e-commerce

Notebook ini membangun prototype aplikasi semantic search yang terintegrasi dengan LLM menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

| Tahap | Implementasi |
|---|---|
| Ingest | `pandas` membaca `data/ecommerce_faq_knowledge_base.csv` (read-only), lalu menggabungkan kategori + question + answer menjadi satu unit retrievable |
| Embed | Gemini Embeddings API (`gemini-embedding-001`, `task_type=RETRIEVAL_DOCUMENT` / `RETRIEVAL_QUERY`) |
| Store | FAISS `IndexIDMap(IndexFlatIP)` pada vektor yang dinormalisasi L2, sehingga skor = cosine similarity. Index disimpan ke `vector_store/` |
| Retrieve | `semantic_search(query, top_k)` → top-k FAQ + cosine similarity |
| Generate | Prompt terstruktur (Persona, Task, Context, Format) → Gemini (`gemini-3.8-flash`) |

**Prasyarat:** environment variable `GEMINI_API_KEY` (atau file `.env` di root project). Lihat `README.md`.

## 1. Import Library & Load Knowledge Base

In [1]:
import json
import os
import re
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

pd.set_option('display.max_colwidth', 120)

# Notebook bisa dijalankan dari root project maupun dari folder notebook/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
VECTOR_STORE_DIR = PROJECT_ROOT / 'vector_store'
DOCS_DIR = PROJECT_ROOT / 'documentation'

load_dotenv(PROJECT_ROOT / '.env')
api_key = os.environ.get('GEMINI_API_KEY')
assert api_key, 'GEMINI_API_KEY belum di-set. Isi di environment variable atau file .env (lihat .env.example).'

EMBEDDING_MODEL = os.environ.get('EMBEDDING_MODEL', 'gemini-embedding-001')
LLM_MODEL = os.environ.get('LLM_MODEL', 'gemini-3.8-flash')

client = genai.Client(api_key=api_key)
print('Embedding model:', EMBEDDING_MODEL)
print('LLM model      :', LLM_MODEL)

Embedding model: gemini-embedding-001
LLM model      : gemini-3.8-flash


In [2]:
def with_retry(fn, *args, retries=5, **kwargs):
    """Panggil Gemini API dengan exponential backoff untuk rate limit / server error sementara."""
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < retries - 1:
                wait = 2 ** attempt * 5
                print(f'  API error {e.code}, retry dalam {wait} detik...')
                time.sleep(wait)
            else:
                raise

In [3]:
# Knowledge base dibaca apa adanya (read-only); semua transformasi dilakukan di memori.
df = pd.read_csv(DATA_PATH)
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform ini?,"Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pe..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,"Saat ini pemesanan hanya bisa dilakukan setelah membuat akun, agar Anda dapat melacak status pesanan dengan mudah."
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah dibuat?,Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Setelah status berubah menja...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesanan diproses penjual?,Penjual biasanya memproses pesanan dalam waktu 1x24 jam setelah pembayaran dikonfirmasi.
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?,Perubahan alamat hanya bisa dilakukan sebelum pesanan berstatus 'Dikirim'. Silakan hubungi customer service secepatn...


### Pemeriksaan struktur data
Cek kolom, tipe data, missing value, duplikasi, distribusi kategori, dan panjang teks. Langkah ini memastikan setiap baris layak menjadi satu unit retrievable.

In [4]:
print('Kolom   :', list(df.columns))
print('Shape   :', df.shape)
print('Missing :', df.isna().sum().to_dict())
print('faq_id unik      :', df['faq_id'].is_unique)
print('Question duplikat:', df['question'].duplicated().sum())
print()
print(df.dtypes)

Kolom   : ['faq_id', 'category', 'question', 'answer']
Shape   : (59, 4)
Missing : {'faq_id': 0, 'category': 0, 'question': 0, 'answer': 0}
faq_id unik      : True
Question duplikat: 0

faq_id      int64
category      str
question      str
answer        str
dtype: object


In [5]:
category_counts = df['category'].value_counts().rename_axis('category').reset_index(name='jumlah_faq')
print('Jumlah kategori:', df['category'].nunique())
category_counts

Jumlah kategori: 8


,category,jumlah_faq
0,Pemesanan,9
1,Pengiriman,9
2,Pembayaran,8
3,Pengembalian,8
4,Akun,7
5,Produk,7
6,Promo,6
7,Customer Service,5


In [6]:
text_len = pd.DataFrame({
    'question_chars': df['question'].str.len(),
    'answer_chars': df['answer'].str.len(),
})
text_len.describe().round(1)

,question_chars,answer_chars
count,59.0,59.0
mean,55.0,128.1
std,10.0,17.4
min,35.0,88.0
25%,46.0,118.0
50%,57.0,127.0
75%,62.0,137.0
max,82.0,181.0


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

Pertimbangan desain:
- **Question + answer digabung.** Pertanyaan pengguna sering mirip dengan *question*, tetapi kadang kata kuncinya hanya ada di *answer* (misal "virtual account" hanya muncul di jawaban FAQ #53).
- **Kategori ikut disertakan** sebagai sinyal topik ringan ("Pengembalian", "Pembayaran", dan sebagainya).
- **Tanpa chunking.** Jawaban terpanjang hanya sekitar 180 karakter, jadi satu FAQ = satu unit.

In [7]:
def build_document(row):
    return (
        f"Kategori: {row['category']}\n"
        f"Pertanyaan: {row['question']}\n"
        f"Jawaban: {row['answer']}"
    )

df['content'] = df.apply(build_document, axis=1)
documents = df['content'].tolist()
print('Jumlah unit retrievable:', len(documents))
print('Contoh unit retrievable:')
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:
Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

- Model `gemini-embedding-001`. Model `text-embedding-004` di starter notebook sudah tidak tersedia di API.
- **Asymmetric task type**: dokumen di-embed dengan `RETRIEVAL_DOCUMENT`, query dengan `RETRIEVAL_QUERY`. Model mengoptimalkan representasi untuk pasangan pertanyaan pendek ↔ dokumen jawaban.
- Embedding dikirim secara **batch** (≤100 teks per request), sehingga 59 FAQ cukup 1 request.
- Vektor dinormalisasi L2 agar inner product = cosine similarity.

In [8]:
BATCH_SIZE = 100

def embed_texts(texts, task_type):
    """Embed list teks dengan Gemini; mengembalikan array float32 yang sudah dinormalisasi L2."""
    vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        result = with_retry(
            client.models.embed_content,
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        vectors.extend(e.values for e in result.embeddings)
    vectors = np.array(vectors, dtype='float32')
    faiss.normalize_L2(vectors)
    return vectors

def get_embedding(text):
    """Embedding untuk satu query pengguna, shape (1, dim)."""
    return embed_texts([text], task_type='RETRIEVAL_QUERY')

embeddings = embed_texts(documents, task_type='RETRIEVAL_DOCUMENT')
print('Shape embeddings:', embeddings.shape)

  API error 429, retry dalam 5 detik...


  API error 429, retry dalam 10 detik...


  API error 429, retry dalam 20 detik...


Shape embeddings: (59, 3072)


In [9]:
# Verifikasi sebelum masuk ke vector database
n_docs, dimension = embeddings.shape
assert n_docs == len(df), 'Jumlah embedding tidak sama dengan jumlah entri FAQ'
assert np.isfinite(embeddings).all(), 'Ada nilai NaN/inf di embedding'
norms = np.linalg.norm(embeddings, axis=1)
print(f'Jumlah vektor : {n_docs} (sama dengan jumlah FAQ: {n_docs == len(df)})')
print(f'Dimensi vektor: {dimension}')
print(f'Norma L2      : min={norms.min():.4f}, max={norms.max():.4f} (≈1 setelah normalisasi)')
print('5 nilai pertama vektor FAQ #1:', np.round(embeddings[0][:5], 4))

Jumlah vektor : 59 (sama dengan jumlah FAQ: True)
Dimensi vektor: 3072
Norma L2      : min=1.0000, max=1.0000 (≈1 setelah normalisasi)
5 nilai pertama vektor FAQ #1: [-0.012   0.0063  0.0214 -0.0712 -0.0025]


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

`IndexFlatIP` melakukan exact search berbasis inner product. Karena vektor sudah dinormalisasi, skornya adalah **cosine similarity** (makin tinggi makin relevan). Index dibungkus `IndexIDMap` supaya ID yang dikembalikan FAISS = `faq_id` asli, bukan posisi baris, sehingga lebih aman jika urutan data berubah.

Index dan metadata-nya disimpan ke `vector_store/` agar bisa dimuat ulang tanpa memanggil Embeddings API lagi.

In [10]:
index = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
index.add_with_ids(embeddings, df['faq_id'].to_numpy(dtype='int64'))

print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df), 'Jumlah vektor di FAISS tidak sesuai jumlah entri knowledge base'
print('Verifikasi OK: jumlah vektor di index = jumlah entri knowledge base')

Jumlah vektor tersimpan di index: 59
Verifikasi OK: jumlah vektor di index = jumlah entri knowledge base


In [11]:
VECTOR_STORE_DIR.mkdir(exist_ok=True)
index_path = VECTOR_STORE_DIR / 'faq_index.faiss'
faiss.write_index(index, str(index_path))

metadata = {
    'embedding_model': EMBEDDING_MODEL,
    'task_type': 'RETRIEVAL_DOCUMENT',
    'dimension': int(dimension),
    'n_vectors': int(index.ntotal),
    'metric': 'inner_product_on_l2_normalized (cosine similarity)',
    'id_field': 'faq_id',
    'source': 'data/ecommerce_faq_knowledge_base.csv',
}
(VECTOR_STORE_DIR / 'index_metadata.json').write_text(json.dumps(metadata, indent=2))

# Sanity check: index yang dimuat ulang dari disk harus identik
reloaded = faiss.read_index(str(index_path))
print('Index tersimpan di :', index_path.relative_to(PROJECT_ROOT))
print('Vektor di file     :', reloaded.ntotal)

Index tersimpan di : vector_store\faq_index.faiss
Vektor di file     : 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

Query di-embed dengan `RETRIEVAL_QUERY`, lalu dicari ke FAISS. Hasilnya berupa daftar FAQ beserta **cosine similarity** (rentang praktis 0–1; makin tinggi makin mirip).

In [12]:
faq_by_id = df.set_index('faq_id')

def semantic_search(query, top_k=3):
    query_embedding = get_embedding(query)
    scores, ids = index.search(query_embedding, top_k)

    results = []
    for rank, (faq_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
        row = faq_by_id.loc[faq_id]
        results.append({
            'rank': rank,
            'faq_id': int(faq_id),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'score': round(float(score), 4),
        })
    return results

def show_results(query, results):
    print(f'Query: {query}')
    print(pd.DataFrame(results)[['rank', 'faq_id', 'score', 'category', 'question']].to_string(index=False))

# Contoh percobaan
show_results('kenapa paket saya belum sampai?', semantic_search('kenapa paket saya belum sampai?'))

Query: kenapa paket saya belum sampai?
 rank  faq_id  score   category                                                          question
    1      12 0.7971 Pengiriman         Bagaimana jika paket saya belum sampai melebihi estimasi?
    2      14 0.7816 Pengiriman Kenapa status pengiriman saya tidak berubah selama beberapa hari?
    3      52 0.7593 Pengiriman       Kenapa nomor resi saya belum bisa dilacak di website kurir?


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

Keputusan desain prompt:
- **Persona** dikirim lewat `system_instruction` agar konsisten di setiap panggilan.
- **Task** berisi aturan grounding eksplisit: hanya boleh memakai CONTEXT, tidak boleh menambah fakta dari pengetahuan umum, dan harus mengabaikan FAQ yang tidak relevan (retrieval selalu mengembalikan top-k meskipun tidak ada yang cocok).
- **Context** memuat ID, kategori, skor similarity, question, dan answer tiap FAQ, dibungkus delimiter `<faq>` agar batas antar-sumber jelas.
- **Format** menentukan bahasa, panjang jawaban, sitasi `[FAQ #id]`, dan kalimat fallback yang baku jika informasi tidak tersedia.

In [13]:
SYSTEM_INSTRUCTION = (
    'Kamu adalah "Asisten CS", asisten customer service virtual untuk sebuah platform e-commerce di Indonesia. '
    'Kamu ramah, sopan, dan to the point. Kamu hanya mengetahui informasi yang ada di FAQ resmi platform '
    'yang diberikan di bagian CONTEXT; kamu tidak memiliki akses ke data pesanan, akun, atau sistem internal pengguna.'
)

FALLBACK_SENTENCE = 'Maaf, informasi tersebut belum tersedia di FAQ kami.'

PROMPT_TEMPLATE = '''### PERSONA
Kamu adalah Asisten CS: customer service e-commerce yang ramah, empatik, dan ringkas.

### TASK
Jawab PERTANYAAN PENGGUNA hanya berdasarkan FAQ di bagian CONTEXT.
Aturan:
1. Gunakan HANYA informasi dari CONTEXT. Jangan menambahkan fakta, angka, kebijakan, nama fitur, atau langkah yang tidak tertulis di CONTEXT, meskipun kamu tahu dari pengetahuan umum.
2. CONTEXT adalah hasil pencarian otomatis dan bisa berisi FAQ yang tidak relevan. Gunakan hanya FAQ yang benar-benar menjawab pertanyaan; abaikan sisanya.
3. Jika CONTEXT hanya menjawab sebagian pertanyaan, jawab bagian yang didukung CONTEXT dan sebutkan dengan jelas bagian mana yang tidak tersedia di FAQ.
4. Jika tidak ada FAQ yang menjawab pertanyaan, mulai jawaban dengan kalimat persis: "{fallback}" Lalu sarankan pengguna menghubungi customer service. Jangan menebak.

### CONTEXT
{context}

### FORMAT
- Bahasa Indonesia yang natural dan sopan, sapa pengguna dengan "Anda".
- Maksimal 4 kalimat, atau langkah bernomor jika jawabannya berupa prosedur.
- Akhiri setiap jawaban yang berasal dari FAQ dengan sumber dalam format [FAQ #id], misalnya [FAQ #12].
- Jangan menyebut kata "CONTEXT", "skor", atau menjelaskan proses pencarian kepada pengguna.

### PERTANYAAN PENGGUNA
{query}

### JAWABAN
'''

def format_context(search_results):
    blocks = []
    for r in search_results:
        blocks.append(
            '<faq id="{faq_id}" kategori="{category}" similarity="{score:.3f}">\n'
            'Pertanyaan: {question}\n'
            'Jawaban: {answer}\n'
            '</faq>'.format(**r)
        )
    return '\n\n'.join(blocks)

def build_prompt(query, search_results):
    return PROMPT_TEMPLATE.format(
        fallback=FALLBACK_SENTENCE,
        context=format_context(search_results),
        query=query.strip(),
    )

# Pratinjau prompt lengkap untuk satu query
print(build_prompt('apakah bisa bayar pakai cicilan?', semantic_search('apakah bisa bayar pakai cicilan?')))

### PERSONA
Kamu adalah Asisten CS: customer service e-commerce yang ramah, empatik, dan ringkas.

### TASK
Jawab PERTANYAAN PENGGUNA hanya berdasarkan FAQ di bagian CONTEXT.
Aturan:
1. Gunakan HANYA informasi dari CONTEXT. Jangan menambahkan fakta, angka, kebijakan, nama fitur, atau langkah yang tidak tertulis di CONTEXT, meskipun kamu tahu dari pengetahuan umum.
2. CONTEXT adalah hasil pencarian otomatis dan bisa berisi FAQ yang tidak relevan. Gunakan hanya FAQ yang benar-benar menjawab pertanyaan; abaikan sisanya.
3. Jika CONTEXT hanya menjawab sebagian pertanyaan, jawab bagian yang didukung CONTEXT dan sebutkan dengan jelas bagian mana yang tidak tersedia di FAQ.
4. Jika tidak ada FAQ yang menjawab pertanyaan, mulai jawaban dengan kalimat persis: "Maaf, informasi tersebut belum tersedia di FAQ kami." Lalu sarankan pengguna menghubungi customer service. Jangan menebak.

### CONTEXT
<faq id="20" kategori="Pembayaran" similarity="0.807">
Pertanyaan: Apakah saya bisa membayar dengan ci

## 7. Mengintegrasikan LLM

`generate_answer` menyatukan seluruh alur: **query → retrieval (FAISS) → prompt → Gemini → jawaban**. Temperature rendah (0.2) dipakai agar jawaban konsisten dan tidak "kreatif" di luar context.

In [14]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = with_retry(
        client.models.generate_content,
        model=LLM_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.2,
        ),
    )
    return response.text.strip(), search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print(f"- [#{s['faq_id']}] (score {s['score']:.3f}) {s['question']}")

Jawaban LLM:
Halo! Ya, fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank. [FAQ #20]

Sumber FAQ yang digunakan:
- [#20] (score 0.807) Apakah saya bisa membayar dengan cicilan?
- [#17] (score 0.694) Metode pembayaran apa saja yang tersedia?
- [#22] (score 0.688) Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

Setiap query diberi **ground truth** (`expected_ids`), yaitu FAQ yang seharusnya ditemukan, sehingga relevansi retrieval bisa diukur (Hit@1, Hit@3, MRR) dan tidak hanya dinilai secara kualitatif. Ada 8 query:

| Tipe | Jumlah | Tujuan |
|---|---|---|
| `direct` | 2 | Kalimat mirip FAQ asli (baseline) |
| `paraphrase` | 4 | Bahasa sehari-hari/slang, sinonim (*duit*, *retur*, *kata sandi*, *nyicil*), tanpa kata kunci FAQ |
| `out_of_kb` | 2 | Satu di luar domain sama sekali, satu *near-miss* (masih seputar pembayaran, tetapi info COD tidak ada di FAQ) |

In [15]:
test_cases = [
    # 1-2 query yang jelas ada jawabannya (formulasi mirip FAQ)
    {'type': 'direct', 'query': 'Bagaimana cara membatalkan pesanan saya?', 'expected_ids': [3]},
    {'type': 'direct', 'query': 'Metode pembayaran apa saja yang bisa saya gunakan?', 'expected_ids': [17]},
    # Parafrase / sinonim / bahasa sehari-hari
    {'type': 'paraphrase', 'query': 'Duit saya kapan balik kalau barangnya sudah saya retur?', 'expected_ids': [25]},
    {'type': 'paraphrase', 'query': 'Saya nggak ingat kata sandi, gimana caranya bisa masuk lagi?', 'expected_ids': [32]},
    {'type': 'paraphrase', 'query': 'Bisa nyicil nggak kalau belanja di sini?', 'expected_ids': [20]},
    {'type': 'paraphrase', 'query': 'Kiriman saya udah telat seminggu dari perkiraan, harus ngapain?', 'expected_ids': [12]},
    # Tidak ada jawabannya di knowledge base
    {'type': 'out_of_kb', 'query': 'Apakah platform ini juga menjual tiket pesawat dan hotel?', 'expected_ids': []},
    {'type': 'out_of_kb', 'query': 'Apakah bisa bayar di tempat (COD) saat barang sampai?', 'expected_ids': []},
]
test_queries = [t['query'] for t in test_cases]
print('Jumlah query uji:', len(test_queries))

Jumlah query uji: 8


In [16]:
test_log = []
for i, case in enumerate(test_cases, start=1):
    q = case['query']
    answer, sources = generate_answer(q, top_k=3)
    test_log.append({**case, 'answer': answer, 'sources': sources})

    print(f"[{i}] ({case['type']}) Query: {q}")
    print('    Ekspektasi FAQ:', case['expected_ids'] or 'tidak ada di knowledge base')
    for s in sources:
        mark = '✓' if s['faq_id'] in case['expected_ids'] else ' '
        print(f"    {mark} #{s['rank']} [FAQ {s['faq_id']:>2}] score={s['score']:.3f}  {s['question']}")
    print('    Jawaban:', answer.replace('\n', '\n             '))
    print('-' * 110)

[1] (direct) Query: Bagaimana cara membatalkan pesanan saya?
    Ekspektasi FAQ: [3]
    ✓ #1 [FAQ  3] score=0.778  Bagaimana cara membatalkan pesanan yang sudah dibuat?
      #2 [FAQ 24] score=0.718  Bagaimana prosedur pengembalian barang yang tidak sesuai?
      #3 [FAQ 44] score=0.698  Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
    Jawaban: Halo! Anda dapat membatalkan pesanan melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Perlu diketahui bahwa setelah status pesanan berubah menjadi 'Dikirim', pembatalan sudah tidak dapat dilakukan. [FAQ #3]
--------------------------------------------------------------------------------------------------------------


  API error 503, retry dalam 5 detik...


[2] (direct) Query: Metode pembayaran apa saja yang bisa saya gunakan?
    Ekspektasi FAQ: [17]
    ✓ #1 [FAQ 17] score=0.788  Metode pembayaran apa saja yang tersedia?
      #2 [FAQ 22] score=0.727  Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
      #3 [FAQ 20] score=0.700  Apakah saya bisa membayar dengan cicilan?
    Jawaban: Halo! Metode pembayaran yang dapat Anda gunakan meliputi transfer bank, kartu kredit/debit, e-wallet, serta pembayaran melalui minimarket rekanan [FAQ #17].
--------------------------------------------------------------------------------------------------------------


[3] (paraphrase) Query: Duit saya kapan balik kalau barangnya sudah saya retur?
    Ekspektasi FAQ: [25]
    ✓ #1 [FAQ 25] score=0.786  Berapa lama proses refund setelah pengembalian disetujui?
      #2 [FAQ 27] score=0.732  Ke mana dana refund akan dikembalikan?
      #3 [FAQ 24] score=0.729  Bagaimana prosedur pengembalian barang yang tidak sesuai?
    Jawaban: Halo! Dana refund Anda biasanya akan diproses dalam waktu 3-7 hari kerja setelah barang yang dikembalikan diterima dan diverifikasi oleh penjual. [FAQ #25]
--------------------------------------------------------------------------------------------------------------


  API error 503, retry dalam 5 detik...


...

[7] (out_of_kb) Query: Apakah platform ini juga menjual tiket pesawat dan hotel?
    Ekspektasi FAQ: tidak ada di knowledge base
      #1 [FAQ  1] score=0.677  Bagaimana cara melakukan pemesanan di platform ini?
      #2 [FAQ 17] score=0.667  Metode pembayaran apa saja yang tersedia?
      #3 [FAQ  2] score=0.659  Apakah saya bisa memesan tanpa membuat akun?
    Jawaban: Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi customer service kami untuk mendapatkan informasi dan bantuan lebih lanjut mengenai hal tersebut.
--------------------------------------------------------------------------------------------------------------


  API error 503, retry dalam 5 detik...


  API error 503, retry dalam 10 detik...


  API error 503, retry dalam 20 detik...


[8] (out_of_kb) Query: Apakah bisa bayar di tempat (COD) saat barang sampai?
    Ekspektasi FAQ: tidak ada di knowledge base
      #1 [FAQ 17] score=0.691  Metode pembayaran apa saja yang tersedia?
      #2 [FAQ 11] score=0.688  Apakah tersedia opsi pengiriman same-day?
      #3 [FAQ  1] score=0.686  Bagaimana cara melakukan pemesanan di platform ini?
    Jawaban: Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan menghubungi tim customer service kami untuk memastikan ketersediaan metode pembayaran di tempat (COD).
--------------------------------------------------------------------------------------------------------------


### Evaluasi kuantitatif retrieval

- **Hit@1 / Hit@3**: apakah FAQ yang diharapkan muncul di peringkat 1 / top-3.
- **MRR** (Mean Reciprocal Rank): 1/peringkat FAQ yang benar, dirata-rata.
- **Gap top1-top2**: selisih skor antara peringkat 1 dan 2. Gap besar berarti retrieval "yakin".
- **Refusal**: apakah LLM menjawab dengan kalimat fallback (diharapkan hanya untuk query `out_of_kb`).

In [17]:
def rank_of_expected(sources, expected_ids):
    for s in sources:
        if s['faq_id'] in expected_ids:
            return s['rank']
    return None

rows = []
for t in test_log:
    rank = rank_of_expected(t['sources'], t['expected_ids'])
    s = t['sources']
    rows.append({
        'type': t['type'],
        'query': t['query'],
        'expected': t['expected_ids'] or '-',
        'top1_faq': s[0]['faq_id'],
        'top1_score': s[0]['score'],
        'gap_top1_top2': round(s[0]['score'] - s[1]['score'], 4),
        'rank_expected': rank if t['expected_ids'] else '-',
        'hit@1': (rank == 1) if t['expected_ids'] else '-',
        'hit@3': (rank is not None) if t['expected_ids'] else '-',
        'llm_refused': t['answer'].startswith(FALLBACK_SENTENCE[:30]),
    })
eval_df = pd.DataFrame(rows)
eval_df

,type,query,expected,top1_faq,top1_score,gap_top1_top2,rank_expected,hit@1,hit@3,llm_refused
0,direct,Bagaimana cara membatalkan pesanan saya?,[3],3,0.7777,0.0596,1,True,True,False
1,direct,Metode pembayaran apa saja yang bisa saya gunakan?,[17],17,0.7883,0.0617,1,True,True,False
2,paraphrase,Duit saya kapan balik kalau barangnya sudah saya retur?,[25],25,0.7856,0.0534,1,True,True,False
3,paraphrase,"Saya nggak ingat kata sandi, gimana caranya bisa masuk lagi?",[32],32,0.7573,0.0568,1,True,True,False
4,paraphrase,Bisa nyicil nggak kalau belanja di sini?,[20],20,0.7974,0.0971,1,True,True,False
5,paraphrase,"Kiriman saya udah telat seminggu dari perkiraan, harus ngapain?",[12],12,0.7858,0.0208,1,True,True,False
6,out_of_kb,Apakah platform ini juga menjual tiket pesawat dan hotel?,-,1,0.6773,0.0106,-,-,-,True
7,out_of_kb,Apakah bisa bayar di tempat (COD) saat barang sampai?,-,17,0.6911,0.0030,-,-,-,True


In [18]:
in_kb = eval_df[eval_df['type'] != 'out_of_kb']
ranks = [r for r in in_kb['rank_expected']]
hit1 = np.mean([r == 1 for r in ranks])
hit3 = np.mean([r is not None for r in ranks])
mrr = np.mean([1 / r if r else 0 for r in ranks])

print(f'Query dengan jawaban di KB: {len(in_kb)}')
print(f'Hit@1 = {hit1:.2f} | Hit@3 = {hit3:.2f} | MRR = {mrr:.3f}')
print()
print('Rata-rata skor top-1 per tipe query:')
print(eval_df.groupby('type')['top1_score'].agg(['mean', 'min', 'max']).round(3))
print()
correct_refusal = (eval_df['llm_refused'] == (eval_df['type'] == 'out_of_kb')).mean()
print(f'Perilaku refusal LLM sesuai ekspektasi: {correct_refusal:.0%} dari query')

Query dengan jawaban di KB: 6
Hit@1 = 1.00 | Hit@3 = 1.00 | MRR = 1.000

Rata-rata skor top-1 per tipe query:
             mean    min    max
type                           
direct      0.783  0.778  0.788
out_of_kb   0.684  0.677  0.691
paraphrase  0.782  0.757  0.797

Perilaku refusal LLM sesuai ekspektasi: 100% dari query


### Pembanding: keyword search sederhana

Untuk membuktikan bahwa sistem menangkap **makna** dan bukan sekadar kata yang sama, query yang sama dicari dengan baseline lexical sederhana: overlap token (setelah stopword dibuang) antara query dan teks FAQ. Jika semantic search menang pada query parafrase, berarti embedding memberi nilai tambah dibanding pencocokan kata.

In [19]:
STOPWORDS = {
    'apa', 'apakah', 'bagaimana', 'cara', 'saya', 'yang', 'di', 'ke', 'dan', 'atau', 'untuk', 'dengan',
    'ini', 'itu', 'bisa', 'sudah', 'kalau', 'jika', 'dari', 'nggak', 'gimana', 'saja', 'juga', 'ada',
    'kapan', 'harus', 'caranya', 'lagi', 'udah', 'sini', 'saat', 'the', 'q', 'a', 'kategori', 'pertanyaan', 'jawaban',
}

def tokenize(text):
    return {t for t in re.findall(r'[a-z0-9]+', text.lower()) if t not in STOPWORDS and len(t) > 1}

doc_tokens = [tokenize(d) for d in documents]

def keyword_search(query, top_k=3):
    q = tokenize(query)
    scores = np.array([len(q & d) / (len(q) or 1) for d in doc_tokens])
    order = np.argsort(-scores, kind='stable')[:top_k]
    return [(int(df.iloc[i]['faq_id']), round(float(scores[i]), 3)) for i in order]

cmp_rows = []
for t in test_log:
    kw = keyword_search(t['query'])
    kw_rank = next((r for r, (fid, sc) in enumerate(kw, 1) if fid in t['expected_ids'] and sc > 0), None)
    sem_rank = rank_of_expected(t['sources'], t['expected_ids'])
    cmp_rows.append({
        'type': t['type'],
        'query': t['query'],
        'expected': t['expected_ids'] or '-',
        'keyword_top1 (faq, overlap)': kw[0],
        'keyword_rank_expected': kw_rank if t['expected_ids'] else '-',
        'semantic_top1': t['sources'][0]['faq_id'],
        'semantic_rank_expected': sem_rank if t['expected_ids'] else '-',
    })
pd.DataFrame(cmp_rows)

,type,query,expected,"keyword_top1 (faq, overlap)",keyword_rank_expected,semantic_top1,semantic_rank_expected
0,direct,Bagaimana cara membatalkan pesanan saya?,[3],"(3, 1.0)",1,3,1
1,direct,Metode pembayaran apa saja yang bisa saya gunakan?,[17],"(1, 0.667)",2,17,1
2,paraphrase,Duit saya kapan balik kalau barangnya sudah saya retur?,[25],"(1, 0.0)",None,25,1
3,paraphrase,"Saya nggak ingat kata sandi, gimana caranya bisa masuk lagi?",[32],"(41, 0.25)",None,32,1
4,paraphrase,Bisa nyicil nggak kalau belanja di sini?,[20],"(42, 0.5)",None,20,1
5,paraphrase,"Kiriman saya udah telat seminggu dari perkiraan, harus ngapain?",[12],"(1, 0.0)",None,12,1
6,out_of_kb,Apakah platform ini juga menjual tiket pesawat dan hotel?,-,"(1, 0.2)",-,1,-
7,out_of_kb,Apakah bisa bayar di tempat (COD) saat barang sampai?,-,"(9, 0.2)",-,17,-


In [20]:
# Simpan hasil mentah pengujian sebagai lampiran dokumentasi
DOCS_DIR.mkdir(exist_ok=True)
raw_path = DOCS_DIR / 'test_run_raw.json'
raw_path.write_text(json.dumps({
    'embedding_model': EMBEDDING_MODEL,
    'llm_model': LLM_MODEL,
    'metrics': {'hit@1': hit1, 'hit@3': hit3, 'mrr': mrr},
    'results': test_log,
    'keyword_baseline': cmp_rows,
}, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Hasil mentah disimpan ke', raw_path.relative_to(PROJECT_ROOT))

Hasil mentah disimpan ke documentation\test_run_raw.json


### Catatan Analisis Relevansi

**Ringkasan:** Hit@1 = Hit@3 = 1.00 (MRR 1.0) pada 6 query yang ada jawabannya, dan LLM menolak dengan benar pada 2 query di luar knowledge base. Analisis lengkap ada di `documentation/retrieval_test_results.md`.

**1. Retrieval akurat, termasuk pada parafrase.** Keempat query parafrase sengaja tidak memakai kata kunci FAQ: *duit balik / retur* ↔ *dana refund / pengembalian*, *nggak ingat kata sandi / masuk* ↔ *lupa password / login*, *nyicil* ↔ *cicilan*, *kiriman telat dari perkiraan* ↔ *paket melebihi estimasi*. Semuanya tetap mendapat FAQ yang benar di peringkat 1, dengan skor top-1 rata-rata (0.782) setara query direct (0.783). Sebaliknya, keyword baseline **gagal di keempat query parafrase** (Hit@1 hanya 1/6). Ini bukti bahwa embedding menangkap makna, bukan sekadar kata yang sama.

**2. Yang kurang relevan / rawan.**
- Query #6 (kiriman telat) hampir tertukar: FAQ #12 (0.786) vs #14 *status pengiriman tidak berubah* (0.765), selisih hanya 0.021. Topik bertetangga seperti ini paling rawan salah urut.
- Peringkat 2–3 sering noise. Contoh: "membatalkan pesanan" ikut mengambil FAQ #24 *pengembalian barang*. LLM berhasil mengabaikannya berkat aturan "abaikan FAQ yang tidak relevan" di prompt.

**3. Query di luar knowledge base.** FAISS selalu mengembalikan top-k, jadi untuk "tiket pesawat" dan "COD" yang terambil adalah FAQ generik (pemesanan, metode pembayaran). Skor top-1-nya lebih rendah (0.677–0.691 vs 0.757–0.797) dan gap top-1/top-2 hampir nol (≤0.011). Pemisahnya ada, tetapi margin ~0.07 terlalu tipis untuk threshold yang andal dengan data uji sekecil ini. Karena itu keputusan "tidak tahu" diserahkan ke LLM lewat prompt, dan kedua query tersebut dijawab dengan kalimat fallback + saran menghubungi CS, tanpa mengarang dari pengetahuan umum.

**4. Kualitas jawaban LLM.** Jawaban grounded: semua angka (3–7 hari kerja, >3 hari) persis dari FAQ, setiap jawaban memakai sitasi `[FAQ #id]`, prosedur otomatis dijadikan langkah bernomor, dan nada empatik muncul pada keluhan. Kelemahan: pada near-miss COD, LLM menolak sepenuhnya padahal bisa menyebut metode yang *tersedia* menurut FAQ #17. Jawaban ini aman, tetapi kurang membantu.

**Keterbatasan:** belum ada similarity threshold, set uji hanya 8 query, topik bertetangga rawan tertukar (kandidat perbaikan: re-ranking / hybrid search), dan redaksi jawaban LLM bisa sedikit berbeda antar-run.